# Llibreries

In [1]:
import pandas as pd
import numpy as np
from zipfile import ZipFile
import os
import sys
from pathlib import Path

# Visualització
import matplotlib.pyplot as plt
import seaborn as sns

# Config vis
sns.set_theme()

# Funcions
cwd = os.getcwd()
parent = os.path.abspath(os.path.join(cwd, os.pardir))
sys.path.insert(0, parent)
from src.utils import neteja_noms_columnes

# Dimensions
Carrega del dataset que conté totes les dimensions de les dades

In [2]:
dimensions = pd.read_csv("../data/dimensions/pad_dimensions.csv")
dim_barris = pd.read_csv("../data/dimensions/BarcelonaCiutat_Barris.csv")

# Dades d' habitatge
En aquesta topologia ingestarem dades sobre preus dels lloguers (€/m2) i si és possible nombre d' habitatges d' ús turístic.

In [51]:
with ZipFile("../data/raw/habitatge/Taula estadística - Preu mitjà per superfície (€_m²) del lloguer d'habitatges.zip", "r") as zip_ref:
    zip_ref.extractall("../data/raw/habitatge/preu_lloguer_hab_zip_extract")

In [52]:
df_lloguer_raw = pd.read_csv("../data/raw/habitatge/preu_lloguer_hab_zip_extract/Taula estadística.csv")
df_lloguer_raw.head()

,Territori,Tipus de territori,2015,2020,2021,2022,2023,2024,2025
0,el Raval,Barri,"11,1","14,0","12,9","15,2","15,9","15,9","16,6"
1,el Barri Gòtic,Barri,"11,2","13,6","12,6","15,6","16,4","16,2","16,6"
2,la Barceloneta,Barri,"16,3","17,3","16,0","18,8","19,0","20,0","22,6"
3,"Sant Pere, Santa Caterina i la Ribera",Barri,"12,7","14,6","13,6","16,0","17,4","17,1","18,3"
4,el Fort Pienc,Barri,"10,9","13,9","13,0","14,4","16,1","15,9","15,8"


In [53]:
df_lloguer_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Territori           73 non-null     object
 1   Tipus de territori  73 non-null     object
 2   2015                73 non-null     object
 3   2020                73 non-null     object
 4   2021                73 non-null     object
 5   2022                73 non-null     object
 6   2023                73 non-null     object
 7   2024                73 non-null     object
 8   2025                73 non-null     object
dtypes: object(9)
memory usage: 5.3+ KB


**Observacions:**
- No s' observen nuls a primera instància.
- Formats incorrectes, degut a les comes com a decimals, ho detecta com a texts.

In [55]:
# Eliminem les columnes que no ens interessen i filtrem
df_lloguer_mod = df_lloguer_raw.drop(columns=["Tipus de territori"])

# Converitm temporalment en format llarg
df_lloguer_stacked = df_lloguer_mod.melt(id_vars=["Territori"], var_name="any", value_name="preu_mitja_m2")
df_lloguer_stacked.head()

,Territori,any,preu_mitja_m2
0,el Raval,2015,"11,1"
1,el Barri Gòtic,2015,"11,2"
2,la Barceloneta,2015,"16,3"
3,"Sant Pere, Santa Caterina i la Ribera",2015,"12,7"
4,el Fort Pienc,2015,"10,9"


In [58]:
# Ens interessen només anys 2015 i 2023
df_lloguer_filt = df_lloguer_stacked[df_lloguer_stacked["any"].isin(["2015", "2023"])]

# Convertim preu a numèric
df_lloguer_filt["preu_mitja_m2"] = pd.to_numeric(df_lloguer_filt["preu_mitja_m2"].astype(str).str.replace(",", "."), errors="coerce")
df_lloguer_filt.info()

<class 'pandas.core.frame.DataFrame'>
Index: 146 entries, 0 to 364
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Territori      146 non-null    object 
 1   any            146 non-null    object 
 2   preu_mitja_m2  144 non-null    float64
dtypes: float64(1), object(2)
memory usage: 4.6+ KB


C:\Users\USER\AppData\Local\Temp\ipykernel_29376\102347946.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_lloguer_filt["preu_mitja_m2"] = pd.to_numeric(df_lloguer_filt["preu_mitja_m2"].astype(str).str.replace(",", "."), errors="coerce")


**Observacions:**
- Ha aparegut NaN per no poder convertir a float.

In [60]:
df_lloguer_filt[df_lloguer_filt["preu_mitja_m2"].isna()]

,Territori,any,preu_mitja_m2
338,Can Peguera,2023,NaN
347,Vallbona,2023,NaN


In [61]:
df_lloguer_raw[(df_lloguer_raw["Territori"] == "Vallbona")|(df_lloguer_raw["Territori"] == "Can Peguera")]

,Territori,Tipus de territori,2015,2020,2021,2022,2023,2024,2025
46,Can Peguera,Barri,"5,8",-,-,-,-,"12,5","8,6"
55,Vallbona,Barri,"6,7","7,9","6,0","8,0",-,-,-


**Observacions:**
- Valor = "-" per a 2023 en ambdos.
- Per simplicitat i no perdre dades, imputarem els valors amb la última dada disponible, i per tant farem aproximacions. Per a Can Peguera utilitzarem el valor de 2024 i per Vallbona el valor de 2022.

In [62]:
df_lloguer_filt.loc[df_lloguer_filt['Territori'] == 'Vallbona', 'preu_mitja_m2'] = 8.0
df_lloguer_filt.loc[df_lloguer_filt['Territori'] == 'Can Peguera', 'preu_mitja_m2'] = 12.5

In [63]:
df_lloguer_filt.info()

<class 'pandas.core.frame.DataFrame'>
Index: 146 entries, 0 to 364
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Territori      146 non-null    object 
 1   any            146 non-null    object 
 2   preu_mitja_m2  146 non-null    float64
dtypes: float64(1), object(2)
memory usage: 4.6+ KB


In [64]:
print("Territoris:\n", df_lloguer_filt.Territori.unique())

Territoris:
 ['el Raval' 'el Barri Gòtic' 'la Barceloneta'
 'Sant Pere, Santa Caterina i la Ribera' 'el Fort Pienc'
 'la Sagrada Família' "la Dreta de l'Eixample"
 "l'Antiga Esquerra de l'Eixample" "la Nova Esquerra de l'Eixample"
 'Sant Antoni' 'el Poble Sec - AEI Parc Montjuïc'
 'la Marina del Prat Vermell - AEI Zona Franca' 'la Marina de Port'
 'la Font de la Guatlla' 'Hostafrancs' 'la Bordeta' 'Sants - Badal'
 'Sants' 'les Corts' 'la Maternitat i Sant Ramon' 'Pedralbes'
 'Vallvidrera, el Tibidabo i les Planes' 'Sarrià' 'les Tres Torres'
 'Sant Gervasi - la Bonanova' 'Sant Gervasi - Galvany'
 'el Putxet i el Farró' 'Vallcarca i els Penitents' 'el Coll' 'la Salut'
 'la Vila de Gràcia' "el Camp d'en Grassot i Gràcia Nova"
 'el Baix Guinardó' 'Can Baró' 'el Guinardó' "la Font d'en Fargues"
 'el Carmel' 'la Teixonera' 'Sant Genís dels Agudells' 'Montbau'
 "la Vall d'Hebron" 'la Clota' 'Horta' 'Vilapicina i la Torre Llobeta'
 'Porta' 'el Turó de la Peira' 'Can Peguera' 'la Guineueta' 'Ca

Es poden veure noms que no apareixen a la dimensió de barris. S' han de netejar.

In [66]:
df_lloguer_filt.loc[:, ["clean_territori"]] = df_lloguer_filt["Territori"].apply(
    lambda x: x.split("-")[0].strip().lower().replace(" ", "").replace("-", "") if "AEI" in x else x.strip().lower().replace(" ", "").replace("-", "")
    )
df_lloguer_filt.head()

,Territori,any,preu_mitja_m2,clean_territori
0,el Raval,2015,11.1,elraval
1,el Barri Gòtic,2015,11.2,elbarrigòtic
2,la Barceloneta,2015,16.3,labarceloneta
3,"Sant Pere, Santa Caterina i la Ribera",2015,12.7,"santpere,santacaterinailaribera"
4,el Fort Pienc,2015,10.9,elfortpienc


In [67]:
# Obtenim el codi del barri a partir del nom
dim_barris["nom_barri_adpt"] = dim_barris["nom_barri"].apply(lambda x: x.strip().lower().replace(" ", "").replace("-", ""))

df_lloguer_codis = df_lloguer_filt.merge(dim_barris[["codi_barri", "nom_barri", "nom_barri_adpt"]], left_on="clean_territori", right_on="nom_barri_adpt", how="left")\
    .drop(columns=["nom_barri_adpt", "clean_territori", "Territori", "nom_barri"])

# Mostrem si hi ha valors sense codi de barri
df_lloguer_codis[df_lloguer_codis["codi_barri"].isna()]

,any,preu_mitja_m2,codi_barri


No hi ha barris sense fer match

In [73]:
df_lloguer_23 = df_lloguer_codis[df_lloguer_codis["any"] == "2023"].drop(columns = ["any"])

df_lloguer_23.head()

,preu_mitja_m2,codi_barri
73,15.9,1
74,16.4,2
75,19.0,3
76,17.4,4
77,16.1,5


In [74]:
df_lloguer_15 = df_lloguer_codis[df_lloguer_codis["any"] == "2015"].drop(columns = ["any"])
df_lloguer_15.head()

,preu_mitja_m2,codi_barri
0,11.1,1
1,11.2,2
2,16.3,3
3,12.7,4
4,10.9,5


## Habitatges Turístics


In [15]:
with ZipFile("../data/raw/habitatge/Taula estadística - Nombre d’habitatges d’ús turístic.zip", "r") as zip_ref:
    zip_ref.extractall("../data/raw/habitatge/habitatges_turistics_zip_extract")

In [17]:
df_turistics_raw = pd.read_csv("../data/raw/habitatge/habitatges_turistics_zip_extract/Taula estadística.csv")
df_turistics_raw.head()

,Territori,Tipus de territori,31 Des. 2015,31 Des. 2023
0,el Raval,Barri,180,212.0
1,el Barri Gòtic,Barri,184,201.0
2,la Barceloneta,Barri,69,64.0
3,"Sant Pere, Santa Caterina i la Ribera",Barri,171,171.0
4,el Fort Pienc,Barri,343,333.0


In [18]:
df_turistics_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65 entries, 0 to 64
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Territori           65 non-null     object 
 1   Tipus de territori  65 non-null     object 
 2   31 Des. 2015        65 non-null     object 
 3   31 Des. 2023        65 non-null     float64
dtypes: float64(1), object(3)
memory usage: 2.2+ KB


**Observacions:**
- Les dades de 2015 presenten algun valor nul.
- Format incorrecte per a les dades de 2015.

In [20]:
df_turistics_raw["31 Des. 2015"].unique()

array(['180', '184', '69', '171', '343', '794', '1.747', '727', '428',
       '568', '546', '3', '6', '74', '187', '42', '90', '189', '36', '8',
       '23', '31', '26', '271', '158', '35', '13', '768', '230', '87',
       '54', '2', '9', '-', '4', '1', '5', '27', '21', '11', '270', '50',
       '101', '111', '406', '145', '30'], dtype=object)

Es pot observar com un dels valors és "-".

In [23]:
# Passem a format llarg
df_turistics_melt = df_turistics_raw[["Territori", "31 Des. 2015", "31 Des. 2023"]].melt(["Territori"], var_name="periode", value_name="pisos_turistics")
df_turistics_melt.head()

,Territori,periode,pisos_turistics
0,el Raval,31 Des. 2015,180
1,el Barri Gòtic,31 Des. 2015,184
2,la Barceloneta,31 Des. 2015,69
3,"Sant Pere, Santa Caterina i la Ribera",31 Des. 2015,171
4,el Fort Pienc,31 Des. 2015,343


Operacions a realitzar:
- Eliminar/Imputar l "-"
- Convertir pisos_turistics a int
- Afegir columna Any
- Creuar amb dim barris
- Separar datasets

In [28]:
df_turistics_mods = df_turistics_melt.copy()

# Convertim a numeric
df_turistics_mods["pisos_turistics"] = pd.to_numeric(df_turistics_mods["pisos_turistics"], errors="coerce")

# Extreiem any
df_turistics_mods["any"] = df_turistics_mods["periode"].str.extract(r"(\d{4})")

df_turistics_mods.head()

,Territori,periode,pisos_turistics,any
0,el Raval,31 Des. 2015,180.0,2015
1,el Barri Gòtic,31 Des. 2015,184.0,2015
2,la Barceloneta,31 Des. 2015,69.0,2015
3,"Sant Pere, Santa Caterina i la Ribera",31 Des. 2015,171.0,2015
4,el Fort Pienc,31 Des. 2015,343.0,2015


In [29]:
# creuem amb dim barri
df_turistics_mods["Territori"].unique()

array(['el Raval', 'el Barri Gòtic', 'la Barceloneta',
       'Sant Pere, Santa Caterina i la Ribera', 'el Fort Pienc',
       'la Sagrada Família', "la Dreta de l'Eixample",
       "l'Antiga Esquerra de l'Eixample",
       "la Nova Esquerra de l'Eixample", 'Sant Antoni', 'el Poble-sec',
       'la Marina del Prat Vermell', 'la Marina de Port',
       'la Font de la Guatlla', 'Hostafrancs', 'la Bordeta',
       'Sants - Badal', 'Sants', 'les Corts',
       'la Maternitat i Sant Ramon', 'Pedralbes',
       'Vallvidrera, el Tibidabo i les Planes', 'Sarrià',
       'les Tres Torres', 'Sant Gervasi - la Bonanova',
       'Sant Gervasi - Galvany', 'el Putxet i el Farró',
       'Vallcarca i els Penitents', 'el Coll', 'la Salut',
       'la Vila de Gràcia', "el Camp d'en Grassot i Gràcia Nova",
       'el Baix Guinardó', 'Can Baró', 'el Guinardó',
       "la Font d'en Fargues", 'el Carmel', 'la Teixonera',
       'Sant Genís dels Agudells', "la Vall d'Hebron", 'Horta',
       'Vilapicina i l

- Els barris d'aquest dataset no semblen presentar problemes per al creuament amb la dimensió de barris

In [37]:
df_turistics_codis = df_turistics_mods.merge(dim_barris[["codi_barri", "nom_barri"]], left_on="Territori", right_on="nom_barri", how="left")\
    .drop(columns=["Territori", "nom_barri", "periode"])

# Mostrem si hi ha valors sense codi de barri
df_turistics_codis[df_turistics_codis["codi_barri"].isna()]

,pisos_turistics,any,codi_barri


In [38]:
df_turistics_codis

,pisos_turistics,any,codi_barri
0,180.0,2015,1
1,184.0,2015,2
2,69.0,2015,3
3,171.0,2015,4
4,343.0,2015,5
...,...,...,...
125,125.0,2023,69
126,32.0,2023,70
127,33.0,2023,71
128,14.0,2023,72


In [39]:
df_turistics_23 = df_turistics_codis[df_turistics_codis["any"] == "2023"].drop(columns = ["any"])

df_turistics_23.head()

,pisos_turistics,codi_barri
65,212.0,1
66,201.0,2
67,64.0,3
68,171.0,4
69,333.0,5


In [71]:
df_turistics_15 = df_turistics_codis[df_turistics_codis["any"] == "2015"].drop(columns = ["any"])

df_turistics_15.head()

,pisos_turistics,codi_barri
0,180.0,1
1,184.0,2
2,69.0,3
3,171.0,4
4,343.0,5


# Agregacions
Un cop hem tractat les diferents dades urbanes, procedim a agrupar-les al seu any corresponent.

In [75]:
df_habitatge = dim_barris[["codi_barri"]].copy()

df_habitatge_23 = df_habitatge\
                        .merge(df_lloguer_23, on="codi_barri", how="left")\
                        .merge(df_turistics_23, on="codi_barri", how="left")

df_habitatge_23.head()

,codi_barri,preu_mitja_m2,pisos_turistics
0,1,15.9,212.0
1,2,16.4,201.0
2,3,19.0,64.0
3,4,17.4,171.0
4,5,16.1,333.0


In [76]:
df_habitatge_15 = df_habitatge\
                        .merge(df_lloguer_15, on="codi_barri", how="left")\
                        .merge(df_turistics_15, on="codi_barri", how="left")

df_habitatge_15.head()

,codi_barri,preu_mitja_m2,pisos_turistics
0,1,11.1,180.0
1,2,11.2,184.0
2,3,16.3,69.0
3,4,12.7,171.0
4,5,10.9,343.0


In [77]:
# Guardem els dataframes processats
df_habitatge_23.to_csv("../data/processed/df_habitatge_23.csv", index=False)
df_habitatge_15.to_csv("../data/processed/df_habitatge_15.csv", index=False)